In [1]:
import os
import sys
import torch
import torch.nn as nn
from torchvision.transforms import v2 as transforms
import librosa
import numpy as np
from sklearn.metrics import roc_auc_score
import wandb

In [2]:
generator = torch.Generator().manual_seed(42)
np.random.seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.RandomCrop(size=(128, 256))


class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir, train):
        self.audio_dir = audio_dir
        file_list = os.listdir(audio_dir)

        labels = np.zeros(len(file_list), dtype=int) if train else [1 if el[0] == 'a' else 0 for el in file_list]
        self.labels = torch.tensor(labels, dtype=torch.int8).to(device)

        loads = [librosa.load(os.path.join(audio_dir, el), sr=None) for el in file_list]
        spectrograms = [librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128) for audio, sr in loads]
        spect_dbs = [
            torch.tensor(
                librosa.power_to_db(spec, ref=np.max),
                dtype=torch.float32
            )
            for spec in spectrograms
        ]

        spect_dbs = torch.stack(spect_dbs)

        mean = spect_dbs.mean(dim=0)
        std = spect_dbs.std(dim=0)

        self.spect_dbs = (spect_dbs - mean) / std

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return transform(self.spect_dbs[idx]), self.labels[idx]

In [5]:
train_dataset = AudioDataset('archive/dev_data/dev_data/slider/train', train=True)
test_dataset = AudioDataset('archive/dev_data/dev_data/slider/test', train=False)

In [6]:
train_dataset.spect_dbs.shape

torch.Size([2370, 128, 313])

In [7]:
batch_size = 16

train_set, validation_set = torch.utils.data.random_split(train_dataset, [0.9, 0.1], generator=generator)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)
validation_loader = torch.utils.data.DataLoader(
    validation_set,
    batch_size=batch_size,
    shuffle=False
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [8]:
class CNNAE(nn.Module):
    def __init__(self):
        super(CNNAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),   # (32, 128, 256)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (32, 64, 128)
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # (64, 64, 128)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (64, 32, 64)
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # (128, 32, 64)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (128, 16, 32)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),   # (64, 32, 64)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),    # (32, 64, 128)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2),     # (1, 128, 256)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.squeeze(1)


class C1DNNAE(nn.Module):
    def __init__(self):
        super(C1DNNAE, self).__init__()
        # Encoder: input shape (batch, 128, 256)
        self.encoder = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2),   # (batch, 64, 128)
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Conv1d(64, 32, kernel_size=5, stride=2, padding=2),    # (batch, 32, 64)
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Conv1d(32, 16, kernel_size=5, stride=2, padding=2),    # (batch, 16, 32)
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(16, 32, kernel_size=4, stride=2, padding=1),  # (batch, 32, 64)
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.ConvTranspose1d(32, 64, kernel_size=4, stride=2, padding=1),  # (batch, 64, 128)
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1),  # (batch, 128, 256)
        )

    def forward(self, x):
        # x: (batch, 128, 256) expected
        x = self.encoder(x)
        x = self.decoder(x)
        return x


class C1DNNAE_INV(nn.Module):
    def __init__(self):
        super(C1DNNAE_INV, self).__init__()
        # Encoder: input shape (batch, 256, 128)
        self.encoder = nn.Sequential(
            nn.Conv1d(256, 128, kernel_size=5, stride=2, padding=2),   # (batch, 128, 64)
            nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2),    # (batch, 64, 32)
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=5, stride=2, padding=2),     # (batch, 32, 16)
            nn.ReLU(),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(32, 64, kernel_size=4, stride=2, padding=1),   # (batch, 64, 32)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1),  # (batch, 128, 64)
            nn.ReLU(),
            nn.ConvTranspose1d(128, 256, kernel_size=4, stride=2, padding=1),  # (batch, 256, 128)
        )

    def forward(self, x):
        # x: (batch, 128, 256) expected
        x = x.permute(0, 2, 1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.permute(0, 2, 1)


class LAE(nn.Module):
    def __init__(self):
        super(LAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(128 * 256, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.decoder = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 2048),
            nn.ReLU(),
            nn.Linear(2048, 128 * 256),
            nn.Tanh()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.view(x.size(0), 128, 256)


class LAE2(nn.Module):
    def __init__(self, input_dim):
        super(LAE2, self).__init__()
        self.encoder = nn.Sequential(
            # Layer 1
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Layer 2
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Layer 3
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Layer 4
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Bottleneck layer
            nn.Linear(128, 8),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.decoder = nn.Sequential(
            # Layer 5
            nn.Linear(8, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Layer 6
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Layer 7
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Layer 8
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # Output layer
            nn.Linear(128, input_dim),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

In [9]:
def train_one_epoch(model, data_loader, optimizer, criterion, scheduler):
    model.train()

    total_loss = 0

    for inputs, _ in data_loader:
        inputs = inputs.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        # print(inputs.shape)
        # print(outputs.shape)

        loss = criterion(outputs, inputs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    return total_loss / len(data_loader)


def validate(model, data_loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, _ in data_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            total_loss += loss.item()

    return total_loss / len(data_loader)


def compute_reconstruction_errors(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch, _ in data_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            # print(reconstructed.shape, batch.shape)
            error = torch.mean(((reconstructed - batch) ** 2).reshape(batch.size(0), -1), dim=1)
            # print(error.shape)
            errors.extend(error.cpu().numpy())
    return errors


def test(model, data_loader):
    model.eval()

    errors = compute_reconstruction_errors(model, data_loader)

    roc_auc_scores = roc_auc_score(test_dataset.labels.cpu(), errors)
    print(f"ROC AUC Score test: {roc_auc_scores}")

    return roc_auc_scores

In [10]:
def train(lr, step_size, gamma, epochs, use_wandb=False):
    model = C1DNNAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
        val_loss = validate(model, validation_loader, criterion)
        roc_auc_score = test(model, test_loader)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        if use_wandb:
            wandb.log({
                "train_loss": train_loss,
                "val_loss": val_loss,
                "roc_auc_score": roc_auc_score,
                "epoch": epoch + 1,
            })

    return model

In [ ]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# key = user_secrets.get_secret('wandb-api-key')

# wandb.login(key=key)

# sweep_configuration = {
#     "method": "grid",
#     "metric": {"goal": "maximize", "name": "roc_auc_score"},
#     'name': "convolutional_1d_autoencoder",
#     "parameters": {
#         "lr": {'values': [1e-2, 1e-3, 1e-4]},
#         "step_size": {'values': [3, 5, 7, 10]},
#         "gamma": {'values': [0.01, 0.1, 0.25, 0.5]},
#     },
# }


# def train_wrapper():
#     with wandb.init() as run:
#         train(
#             lr=run.config.lr,
#             step_size=run.config.step_size,
#             gamma=run.config.gamma,
#             epochs=50,
#             use_wandb=True
#         )


# sweep_id = wandb.sweep(sweep=sweep_configuration, entity='matteo-ghia-politecnico-di-torino', project="aml challenge 2")
# print(f"Sweep ID: {sweep_id}")
# wandb.agent(sweep_id, function=train_wrapper)

In [11]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# key = user_secrets.get_secret('wandb-api-key')

# wandb.login(key=key)

# with wandb.init(project="aml-challenge-2", entity="matteo-ghia-politecnico-di-torino", name='Conv1D autoencoder') as run:
model = train(lr=1e-3, gamma=0.5, step_size=10, epochs=100, use_wandb=False)
# torch.save(model.state_dict(), "model.pt")

ROC AUC Score test: 0.5265709529754474
Epoch 1/100, Train Loss: 0.6570, Validation Loss: 0.5580
ROC AUC Score test: 0.5481439866833125
Epoch 2/100, Train Loss: 0.5240, Validation Loss: 0.4651
ROC AUC Score test: 0.579692051602164
Epoch 3/100, Train Loss: 0.4560, Validation Loss: 0.4373
ROC AUC Score test: 0.5825260091552227
Epoch 4/100, Train Loss: 0.4353, Validation Loss: 0.4428
ROC AUC Score test: 0.5959758635039534
Epoch 5/100, Train Loss: 0.4319, Validation Loss: 0.4390
ROC AUC Score test: 0.5820183104452767
Epoch 6/100, Train Loss: 0.4196, Validation Loss: 0.4299
ROC AUC Score test: 0.5881647940074907
Epoch 7/100, Train Loss: 0.4203, Validation Loss: 0.4331
ROC AUC Score test: 0.5906533499791927
Epoch 8/100, Train Loss: 0.4075, Validation Loss: 0.4054
ROC AUC Score test: 0.5945859342488556
Epoch 9/100, Train Loss: 0.3928, Validation Loss: 0.3934
ROC AUC Score test: 0.6087265917602996
Epoch 10/100, Train Loss: 0.3856, Validation Loss: 0.3738
ROC AUC Score test: 0.5984810653349979
E

In [ ]:
# model = C1DNNAE().to(device)
# model.load_state_dict(torch.load("model.pt", map_location=device))